# LeetCode 238 — Product of Array Except Self bằng Python

Link đề: https://leetcode.com/problems/product-of-array-except-self/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def productExceptSelf(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (nhân trực tiếp mọi phần tử khác chỉ số `i`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi chỉ số `i`, `answer[i]` bằng **tích của mọi phần tử của `nums` trừ `nums[i]`**; **không dùng phép chia**, chạy trong `O(n)`.
2. Biết **vì sao phép chia là cái bẫy**: khi mảng có số `0` thì chia cho `0` gây lỗi và phải xử lý riêng (số lượng số `0` quyết định kết quả).
3. Nhận ra **bước đột phá**: `answer[i] = (tích mọi phần tử bên trái i) × (tích mọi phần tử bên phải i)`, mỗi vế tính được bằng **tích tiền tố (prefix)** và **tích hậu tố (suffix)**.
4. Đi từ **brute force `O(n²)`** → **chia cho tích cả mảng (có xử lý số 0)** → **mảng tiền tố + mảng hậu tố (`O(n)` bộ nhớ phụ)** → **hai lượt trên chính mảng kết quả (`O(1)` bộ nhớ phụ, cách nộp bài)** → **một vòng lặp chạy từ hai đầu vào giữa**.
5. Tránh các lỗi kinh điển: chia cho `0`, cập nhật tích **trước** khi gán (làm tích gồm cả chính phần tử đó), khởi tạo tích bằng `0` thay vì `1`, và chạy lượt hậu tố sai chiều.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, trả về mảng `answer` sao cho `answer[i]` bằng **tích của tất cả các phần tử của `nums` ngoại trừ `nums[i]`**.

Tích của **bất kỳ tiền tố hoặc hậu tố nào** của `nums` được đảm bảo vừa trong số nguyên 32 bit.

Bạn phải viết thuật toán chạy trong thời gian `O(n)` và **không dùng phép chia**.

**Ví dụ 1**

- Input: `nums = [1,2,3,4]`
- Output: `[24,12,8,6]`

**Ví dụ 2**

- Input: `nums = [-1,1,0,-3,3]`
- Output: `[0,0,9,0,0]`

**Ràng buộc:** `2 <= nums.length <= 10⁵`; `-30 <= nums[i] <= 30`; tích của mọi tiền tố/hậu tố vừa 32 bit.

**Follow up:** giải với bộ nhớ phụ `O(1)` (mảng kết quả không tính vào bộ nhớ phụ).

In [1]:
from typing import List
import random
from math import prod

examples = [
    ([1, 2, 3, 4], [24, 12, 8, 6]),
    ([-1, 1, 0, -3, 3], [0, 0, 9, 0, 0]),
]
for nums, expected in examples:
    print(f"nums={nums}, kỳ vọng {expected}")

nums=[1, 2, 3, 4], kỳ vọng [24, 12, 8, 6]
nums=[-1, 1, 0, -3, 3], kỳ vọng [0, 0, 9, 0, 0]


## 3. Kiến thức nền

### 3.1. Tích của một dãy: vòng lặp và `math.prod`

Tích của dãy rỗng được quy ước bằng `1` (phần tử trung hòa của phép nhân), giống tổng của dãy rỗng bằng `0`. Có thể tính tích bằng vòng lặp với biến khởi tạo `1`, hoặc `math.prod`.

In [2]:
nums = [1, 2, 3, 4]
product = 1                       # khởi tạo 1 (KHÔNG phải 0)
for x in nums:
    product *= x
print(product, prod(nums), prod([]))      # 24 24 1
print(1 * 5, 0 * 5)                        # 5 0: nhân với 1 giữ nguyên, nhân với 0 triệt tiêu

24 24 1
5 0


### 3.2. Lát cắt và "tích mọi phần tử bên trái / bên phải"

`nums[:i]` là các phần tử **bên trái** vị trí `i` (không gồm `i`), `nums[i + 1:]` là các phần tử **bên phải**. Tích hai bên nhân lại chính là tích mọi phần tử trừ `nums[i]`.

In [3]:
nums = [1, 2, 3, 4]
for i in range(len(nums)):
    left, right = prod(nums[:i]), prod(nums[i + 1:])
    print(f"i={i}: bên trái {nums[:i]} -> {left}, bên phải {nums[i + 1:]} -> {right}, tích = {left * right}")

i=0: bên trái [] -> 1, bên phải [2, 3, 4] -> 24, tích = 24
i=1: bên trái [1] -> 1, bên phải [3, 4] -> 12, tích = 12
i=2: bên trái [1, 2] -> 2, bên phải [4] -> 4, tích = 8
i=3: bên trái [1, 2, 3] -> 6, bên phải [] -> 1, tích = 6


### 3.3. Tích tiền tố (prefix) và hậu tố (suffix)

**Tiền tố** `prefix[i]` = tích `nums[0..i-1]` (tích mọi phần tử bên **trái** `i`), với `prefix[0] = 1`. **Hậu tố** `suffix[i]` = tích `nums[i+1..n-1]` (tích mọi phần tử bên **phải** `i`), với `suffix[n-1] = 1`. Tính dần: `prefix[i] = prefix[i-1] * nums[i-1]`.

In [4]:
nums = [1, 2, 3, 4]
n = len(nums)
prefix = [1] * n
for i in range(1, n):
    prefix[i] = prefix[i - 1] * nums[i - 1]
suffix = [1] * n
for i in range(n - 2, -1, -1):
    suffix[i] = suffix[i + 1] * nums[i + 1]
print("prefix:", prefix)       # [1, 1, 2, 6]
print("suffix:", suffix)       # [24, 12, 4, 1]
print("prefix[i] * suffix[i]:", [a * b for a, b in zip(prefix, suffix)])   # [24, 12, 8, 6]

prefix: [1, 1, 2, 6]
suffix: [24, 12, 4, 1]
prefix[i] * suffix[i]: [24, 12, 8, 6]


### 3.4. Duyệt ngược: `range(n - 1, -1, -1)` và `reversed`

Để duyệt từ phải sang trái: `for i in range(n - 1, -1, -1)` (từ `n−1` về `0`, bước `−1`), hoặc `for x in reversed(nums)`. Nhớ rằng điểm dừng `-1` **không** được duyệt (nên bao gồm `0`).

In [5]:
n = 4
print(list(range(n - 1, -1, -1)))          # [3, 2, 1, 0]
print(list(range(n - 2, -1, -1)))          # [2, 1, 0]
print(list(reversed([10, 20, 30])))        # [30, 20, 10]

[3, 2, 1, 0]
[2, 1, 0]
[30, 20, 10]


### 3.5. Số 0 làm hỏng phép chia

Nếu có `total = tích cả mảng` thì `answer[i] = total // nums[i]` (khi `nums[i] ≠ 0`). Nhưng khi `nums[i] = 0` thì chia cho `0` gây `ZeroDivisionError`; hơn nữa nếu mảng có một số `0` thì `total = 0` và công thức cho sai các vị trí khác. Đề còn **cấm chia**, nên ta cần cách khác.

In [6]:
nums = [-1, 1, 0, -3, 3]
total = prod(nums)
print("tích cả mảng:", total)                       # 0
try:
    print([total // x for x in nums])
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

tích cả mảng: 0
ZeroDivisionError: integer division or modulo by zero


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ `nums = [1, 2, 3, 4]`. Tính tay từng vị trí:

| `i` | bỏ `nums[i]` | tích còn lại |
|---|---|---|
| 0 | `2 · 3 · 4` | `24` |
| 1 | `1 · 3 · 4` | `12` |
| 2 | `1 · 2 · 4` | `8` |
| 3 | `1 · 2 · 3` | `6` |

Nhận xét: với `i = 2`, tích còn lại là `(1 · 2) · (4)`: **tích bên trái** `2` nhân **tích bên phải** `4`. Tương tự `i = 1`: `(1) · (3 · 4)`. Vậy:

```
answer[i] = (tích các phần tử trước i) × (tích các phần tử sau i)
```

Mỗi vế là một **tích tiền tố / hậu tố**, tính dần được trong `O(n)` tổng cộng thay vì nhân lại từ đầu cho mỗi `i`.

In [7]:
nums = [-1, 1, 0, -3, 3]
n = len(nums)
for i in range(n):
    left = prod(nums[:i])
    right = prod(nums[i + 1:])
    print(f"i={i}: trái={left}, phải={right}, answer[{i}] = {left * right}")

i=0: trái=1, phải=0, answer[0] = 0
i=1: trái=-1, phải=0, answer[1] = 0
i=2: trái=-1, phải=-9, answer[2] = 9
i=3: trái=0, phải=3, answer[3] = 0
i=4: trái=0, phải=1, answer[4] = 0


## 5. Cách 1 — Brute Force / baseline: nhân trực tiếp mọi phần tử khác `i`

**Ý tưởng.** Với mỗi `i`, duyệt mọi `j ≠ i` và nhân `nums[j]` lại. Rõ ràng, nhưng làm đi làm lại nhiều phép nhân giống nhau.

In [8]:
def product_except_self_bruteforce(nums: list[int]) -> list[int]:
    n = len(nums)
    answer = []
    for i in range(n):
        product = 1
        for j in range(n):
            if j != i:                 # bỏ chính phần tử i
                product *= nums[j]
        answer.append(product)
    return answer


for nums, expected in examples:
    print(f"nums={nums} -> {product_except_self_bruteforce(nums)} (kỳ vọng {expected})")

nums=[1, 2, 3, 4] -> [24, 12, 8, 6] (kỳ vọng [24, 12, 8, 6])
nums=[-1, 1, 0, -3, 3] -> [0, 0, 9, 0, 0] (kỳ vọng [0, 0, 9, 0, 0])


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Hai vòng lặp lồng nhau: **Time `O(n²)`**, **Space `O(1)`** (không kể mảng kết quả). Với `n = 10⁵` là `10¹⁰` phép nhân: **quá chậm**.

**Nút thắt.** Tích bên trái của `i` và tích bên trái của `i + 1` chỉ khác nhau một thừa số (`nums[i]`), nhưng ta tính lại từ đầu. Cần chia sẻ phép tính.

## 6. Ý tưởng thuật toán / bước đột phá

**Tách tích thành hai nửa.** `answer[i] = L[i] × R[i]`, với `L[i]` là tích mọi phần tử **bên trái** `i` và `R[i]` là tích mọi phần tử **bên phải** `i`. Cả hai được tính **dần**:

```
L[0] = 1,   L[i] = L[i-1] * nums[i-1]
R[n-1] = 1, R[i] = R[i+1] * nums[i+1]
```

Từ đó có ba mức tối ưu bộ nhớ:

1. Lưu hai mảng `L` và `R` rồi nhân từng cặp (`O(n)` bộ nhớ phụ): Cách 3.
2. Chỉ lưu `L` vào chính mảng kết quả, rồi duyệt từ phải sang trái giữ **một biến** `right` là tích hậu tố đang chạy và nhân vào kết quả: **`O(1)` bộ nhớ phụ** (Cách 4).
3. Chạy cả hai chiều trong **một vòng lặp** (Cách 5).

**Không cần chia.** Cách này cũng xử lý tự nhiên mọi số `0` vì ta không bao giờ chia, chỉ nhân.

In [9]:
nums = [-1, 1, 0, -3, 3]
n = len(nums)
L = [1] * n
for i in range(1, n):
    L[i] = L[i - 1] * nums[i - 1]
R = [1] * n
for i in range(n - 2, -1, -1):
    R[i] = R[i + 1] * nums[i + 1]
print("L =", L)
print("R =", R)
print("answer =", [a * b for a, b in zip(L, R)])       # [0, 0, 9, 0, 0]

L = [1, -1, -1, 0, 0]
R = [0, 0, -9, 3, 1]
answer = [0, 0, 9, 0, 0]


## 7. Các cách tiếp theo

### Cách 2 — Chia cho tích cả mảng (CÓ chia: không hợp lệ theo đề, nhưng đáng hiểu)

**Giải thích.** Đếm số `0` trong mảng:

- **không có `0`:** `answer[i] = total // nums[i]` với `total` là tích cả mảng;
- **đúng một `0` (ở vị trí `j`):** mọi vị trí khác bằng `0`; `answer[j]` bằng tích của **các phần tử khác 0**;
- **từ hai `0` trở lên:** mọi `answer[i] = 0`.

Phải dùng phép chia nguyên `//` (chia hết nên chính xác) và xử lý số `0` riêng. Đề cấm phép chia; ta dùng cách này làm **chuẩn đối chiếu** và để thấy độ rườm rà của nó.

In [10]:
def product_except_self_division(nums: list[int]) -> list[int]:
    zeros = nums.count(0)
    total_nonzero = 1
    for x in nums:
        if x != 0:
            total_nonzero *= x                 # tích các phần tử khác 0
    n = len(nums)
    if zeros >= 2:
        return [0] * n
    if zeros == 1:
        return [total_nonzero if x == 0 else 0 for x in nums]
    return [total_nonzero // x for x in nums]  # không có 0: chia hết


for nums, expected in examples:
    print(f"nums={nums} -> {product_except_self_division(nums)} (kỳ vọng {expected})")

nums=[1, 2, 3, 4] -> [24, 12, 8, 6] (kỳ vọng [24, 12, 8, 6])
nums=[-1, 1, 0, -3, 3] -> [0, 0, 9, 0, 0] (kỳ vọng [0, 0, 9, 0, 0])


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`** ngoài mảng kết quả. Nhưng **dùng phép chia** nên không được chấp nhận theo đề, và phải xử lý ba trường hợp số `0`.

### Cách 3 — Mảng tiền tố và mảng hậu tố (`O(n)` bộ nhớ phụ)

**Giải thích.** Dựng `left[i]` (tích bên trái) và `right[i]` (tích bên phải) như ở mục 6; `answer[i] = left[i] * right[i]`. Rõ ràng và không chia, tốn hai mảng phụ.

In [11]:
def product_except_self_arrays(nums: list[int]) -> list[int]:
    n = len(nums)
    left = [1] * n                              # left[i] = tích nums[0..i-1]
    for i in range(1, n):
        left[i] = left[i - 1] * nums[i - 1]
    right = [1] * n                             # right[i] = tích nums[i+1..n-1]
    for i in range(n - 2, -1, -1):
        right[i] = right[i + 1] * nums[i + 1]
    return [left[i] * right[i] for i in range(n)]


for nums, expected in examples:
    print(f"nums={nums} -> {product_except_self_arrays(nums)} (kỳ vọng {expected})")

nums=[1, 2, 3, 4] -> [24, 12, 8, 6] (kỳ vọng [24, 12, 8, 6])
nums=[-1, 1, 0, -3, 3] -> [0, 0, 9, 0, 0] (kỳ vọng [0, 0, 9, 0, 0])


**Độ phức tạp Cách 3:** **Time `O(n)`** (ba lượt), **Space `O(n)`** cho `left` và `right`.

### Cách 4 — Hai lượt trên chính mảng kết quả (`O(1)` bộ nhớ phụ) — cách của lời giải tham khảo

**Giải thích.**

1. **Lượt trái → phải:** giữ biến `left = 1` (tích bên trái). Với mỗi `i`: `ans[i] = left`, **rồi** `left *= nums[i]`. Sau lượt này `ans[i]` là tích bên trái của `i`.
2. **Lượt phải → trái:** giữ biến `right = 1` (tích bên phải). Với mỗi `i` từ cuối về đầu: `ans[i] *= right`, **rồi** `right *= nums[i]`.

Thứ tự "gán trước, nhân biến cập nhật sau" đảm bảo mỗi tích **không gồm** `nums[i]`.

In [12]:
def product_except_self_two_pass(nums: list[int]) -> list[int]:
    n = len(nums)
    ans = [0] * n
    left = 1
    for i in range(n):                           # lượt 1: ans[i] = tích bên trái của i
        ans[i] = left
        left *= nums[i]                          # cập nhật SAU khi gán (không gồm nums[i])
    right = 1
    for i in range(n - 1, -1, -1):               # lượt 2: nhân thêm tích bên phải
        ans[i] *= right
        right *= nums[i]
    return ans


for nums, expected in examples:
    print(f"nums={nums} -> {product_except_self_two_pass(nums)} (kỳ vọng {expected})")

nums=[1, 2, 3, 4] -> [24, 12, 8, 6] (kỳ vọng [24, 12, 8, 6])
nums=[-1, 1, 0, -3, 3] -> [0, 0, 9, 0, 0] (kỳ vọng [0, 0, 9, 0, 0])


In [13]:
# Code nộp LeetCode cho Cách 4
from typing import List


class Solution:
    def productExceptSelf(self, nums: List[int]) -> List[int]:
        n = len(nums)
        ans = [0] * n
        left = right = 1
        for i, x in enumerate(nums):
            ans[i] = left
            left *= x
        for i in range(n - 1, -1, -1):
            ans[i] *= right
            right *= nums[i]
        return ans

**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** bộ nhớ phụ (hai biến; mảng kết quả không tính theo đề).

### Cách 5 — Một vòng lặp chạy từ hai đầu vào giữa

**Giải thích.** Khởi tạo `ans = [1] * n`, `left = right = 1`. Tại bước `i`: gán `ans[i] *= left` (tích bên trái của `i`) và `ans[n−1−i] *= right` (tích bên phải của `n−1−i`), rồi cập nhật `left *= nums[i]`, `right *= nums[n−1−i]`. Mỗi `ans[k]` được nhân đúng hai lần (một lần từ bên trái, một lần từ bên phải) nên không cần hai vòng riêng.

In [14]:
def product_except_self_one_loop(nums: list[int]) -> list[int]:
    n = len(nums)
    ans = [1] * n
    left = right = 1
    for i in range(n):
        j = n - 1 - i                            # chỉ số đối xứng từ bên phải
        ans[i] *= left                           # tích bên trái của i
        ans[j] *= right                          # tích bên phải của j
        left *= nums[i]
        right *= nums[j]
    return ans


for nums, expected in examples:
    print(f"nums={nums} -> {product_except_self_one_loop(nums)} (kỳ vọng {expected})")

nums=[1, 2, 3, 4] -> [24, 12, 8, 6] (kỳ vọng [24, 12, 8, 6])
nums=[-1, 1, 0, -3, 3] -> [0, 0, 9, 0, 0] (kỳ vọng [0, 0, 9, 0, 0])


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`** bộ nhớ phụ.

> Cách 4 là cách nộp bài chuẩn, đáp ứng follow up. Cách 3 dễ hiểu nhất. Cách 2 cho thấy vì sao đề cấm chia. Cách 1 là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 4 trên `nums = [-1, 1, 0, -3, 3]`: bảng `left`, `right`, `ans` qua từng bước của hai lượt.

In [15]:
def trace_two_pass(nums):
    n = len(nums)
    ans = [0] * n
    print("nums =", nums)
    print("-- lượt 1 (trái -> phải): ans[i] = left; rồi left *= nums[i]")
    left = 1
    for i in range(n):
        ans[i] = left
        print(f"  i={i}: gán ans[{i}] = left = {left}; left *= nums[{i}]={nums[i]} -> left = {left * nums[i]}; ans = {ans}")
        left *= nums[i]
    print("-- lượt 2 (phải -> trái): ans[i] *= right; rồi right *= nums[i]")
    right = 1
    for i in range(n - 1, -1, -1):
        ans[i] *= right
        print(f"  i={i}: ans[{i}] *= right={right} -> ans = {ans}; right *= nums[{i}]={nums[i]} -> right = {right * nums[i]}")
        right *= nums[i]
    print("  => đáp án:", ans)
    return ans


assert trace_two_pass([-1, 1, 0, -3, 3]) == [0, 0, 9, 0, 0]

nums = [-1, 1, 0, -3, 3]
-- lượt 1 (trái -> phải): ans[i] = left; rồi left *= nums[i]
  i=0: gán ans[0] = left = 1; left *= nums[0]=-1 -> left = -1; ans = [1, 0, 0, 0, 0]
  i=1: gán ans[1] = left = -1; left *= nums[1]=1 -> left = -1; ans = [1, -1, 0, 0, 0]
  i=2: gán ans[2] = left = -1; left *= nums[2]=0 -> left = 0; ans = [1, -1, -1, 0, 0]
  i=3: gán ans[3] = left = 0; left *= nums[3]=-3 -> left = 0; ans = [1, -1, -1, 0, 0]
  i=4: gán ans[4] = left = 0; left *= nums[4]=3 -> left = 0; ans = [1, -1, -1, 0, 0]
-- lượt 2 (phải -> trái): ans[i] *= right; rồi right *= nums[i]
  i=4: ans[4] *= right=1 -> ans = [1, -1, -1, 0, 0]; right *= nums[4]=3 -> right = 3
  i=3: ans[3] *= right=3 -> ans = [1, -1, -1, 0, 0]; right *= nums[3]=-3 -> right = -9
  i=2: ans[2] *= right=-9 -> ans = [1, -1, 9, 0, 0]; right *= nums[2]=0 -> right = 0
  i=1: ans[1] *= right=0 -> ans = [1, 0, 9, 0, 0]; right *= nums[1]=1 -> right = 0
  i=0: ans[0] *= right=0 -> ans = [0, 0, 9, 0, 0]; right *= nums[0]=-1 -> right = 

Bảng tóm tắt (Cách 4, `nums = [-1, 1, 0, -3, 3]`):

| `i` | `nums[i]` | tích trái `L[i]` | tích phải `R[i]` | `ans[i] = L[i] · R[i]` |
|---|---|---|---|---|
| 0 | `-1` | `1` | `1·0·(-3)·3 = 0` | `0` |
| 1 | `1` | `-1` | `0·(-3)·3 = 0` | `0` |
| 2 | `0` | `-1·1 = -1` | `(-3)·3 = -9` | `9` |
| 3 | `-3` | `-1·1·0 = 0` | `3` | `0` |
| 4 | `3` | `0` | `1` | `0` |

Chỉ vị trí của số `0` (i = 2) có thể khác `0`, bằng tích các số còn lại `(-1)·1·(-3)·3 = 9`. Cả hai lượt đều chỉ **nhân**, không chia, nên số `0` xử lý tự nhiên.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4. Đúng class `Solution`, đúng method `productExceptSelf(self, nums: List[int]) -> List[int]`.

In [16]:
from typing import List


class Solution:
    def productExceptSelf(self, nums: List[int]) -> List[int]:
        n = len(nums)
        ans = [0] * n
        left = 1                         # tích mọi phần tử bên trái vị trí hiện tại
        for i in range(n):
            ans[i] = left                # gán TRƯỚC (không gồm nums[i])
            left *= nums[i]
        right = 1                        # tích mọi phần tử bên phải vị trí hiện tại
        for i in range(n - 1, -1, -1):
            ans[i] *= right              # nhân thêm tích bên phải
            right *= nums[i]
        return ans


for nums, expected in examples:
    assert Solution().productExceptSelf(nums) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Phân rã:** với mỗi `i`, mọi phần tử khác `nums[i]` hoặc nằm bên trái (`j < i`) hoặc bên phải (`j > i`), không trùng và không sót. Tích của chúng là `(Π_{j<i} nums[j]) · (Π_{j>i} nums[j]) = L[i] · R[i]` (phép nhân có tính kết hợp và giao hoán).
- **Bất biến lượt 1:** ngay trước khi xử lý chỉ số `i`, biến `left = Π_{j<i} nums[j]` (`left` bắt đầu bằng `1` = tích rỗng). Ta gán `ans[i] = left`, rồi nhân `nums[i]` để bất biến đúng cho `i + 1`. Vì gán **trước** khi nhân nên `ans[i]` không gồm `nums[i]`.
- **Bất biến lượt 2:** trước khi xử lý `i` (từ phải sang trái), `right = Π_{j>i} nums[j]`. Nhân vào `ans[i]` cho `L[i] · R[i]`; rồi cập nhật `right`.
- **Số 0 và số âm:** chỉ dùng phép nhân nên mọi trường hợp (số `0`, số âm, số `1`) đều đúng; không cần phân biệt trường hợp.
- **Cách 5:** tại bước `i` nhân `left` vào `ans[i]` và `right` vào `ans[n−1−i]`; sau cả vòng, mỗi `ans[k]` nhận đúng một lần tích trái (khi `i = k`) và một lần tích phải (khi `i = n−1−k`) và bắt đầu từ `1`.
- **Cách 2:** đúng nhờ chia ba trường hợp số `0`, nhưng dùng phép chia nên bị cấm.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- **Cách 1 (brute force):** Time `O(n²)`, Space `O(1)`.
- **Cách 2 (chia cho tích, có xử lý 0):** Time `O(n)`, Space `O(1)` (nhưng dùng phép chia, bị cấm).
- **Cách 3 (hai mảng tiền tố/hậu tố):** Time `O(n)`, Space `O(n)`.
- **Cách 4 (hai lượt trên mảng kết quả):** Time `O(n)`, Space `O(1)` bộ nhớ phụ.
- **Cách 5 (một vòng hai đầu):** Time `O(n)`, Space `O(1)` bộ nhớ phụ.
- Trường hợp trung bình và xấu nhất như nhau. Mỗi phép nhân là `O(1)` nhờ đảm bảo tích vừa 32 bit (với số nguyên lớn của Python thì giá trị lớn tốn thêm chút thời gian nhưng không xảy ra theo ràng buộc).

## 12. Bộ test

Ta gom các cách vào từ điển `impls`. `oracle` là phép nhân trực tiếp mọi `nums[j]` với `j ≠ i`, dùng `math.prod` trên danh sách đã loại phần tử `i` (hoàn toàn khác tiền tố/hậu tố).

In [17]:
sol = Solution()
impls = {
    "bruteforce": product_except_self_bruteforce,
    "division": product_except_self_division,
    "arrays": product_except_self_arrays,
    "two_pass": product_except_self_two_pass,
    "one_loop": product_except_self_one_loop,
    "submit": sol.productExceptSelf,
}


def oracle(nums):
    return [prod(nums[:i] + nums[i + 1:]) for i in range(len(nums))]


def run_all(nums):
    results = {}
    for name, fn in impls.items():
        original = list(nums)
        results[name] = fn(original)
        assert original == nums, (name, "đã sửa đầu vào")
    return results


for nums, expected in examples:
    assert oracle(nums) == expected
    for name, got in run_all(nums).items():
        assert got == expected, (name, nums, got, expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [18]:
def check_all(nums, expected, label):
    assert oracle(nums) == expected, (label, "oracle", oracle(nums), expected)
    for name, got in run_all(nums).items():
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected if len(expected) <= 8 else f"[... {len(expected)} phần tử]")


check_all([3, 4], [4, 3], "n = 2")
check_all([0, 0], [0, 0], "n = 2, hai số 0")
check_all([0, 5], [5, 0], "n = 2, một số 0")
check_all([1, 1, 1, 1], [1, 1, 1, 1], "toàn số 1")
check_all([-1, -2, -3], [6, 3, 2], "toàn số âm (tích hai số âm dương)")
check_all([2, 0, 3, 0], [0, 0, 0, 0], "hai số 0: mọi phần tử bằng 0")
check_all([2, 0, 3], [0, 6, 0], "một số 0: chỉ vị trí số 0 khác 0")
check_all([30, -30, 1], [-30, 30, -900], "giá trị biên -30, 30")
check_all([-1, 1, 0, -3, 3], [0, 0, 9, 0, 0], "ví dụ 2 của đề")
check_all([1, 2, 3, 4, 5], [120, 60, 40, 30, 24], "1..5")
check_all([5, 1, 1, 1, 1, 1], [1, 5, 5, 5, 5, 5], "một số lớn, còn lại là 1")

PASS n = 2 -> [4, 3]
PASS n = 2, hai số 0 -> [0, 0]
PASS n = 2, một số 0 -> [5, 0]
PASS toàn số 1 -> [1, 1, 1, 1]
PASS toàn số âm (tích hai số âm dương) -> [6, 3, 2]
PASS hai số 0: mọi phần tử bằng 0 -> [0, 0, 0, 0]
PASS một số 0: chỉ vị trí số 0 khác 0 -> [0, 6, 0]
PASS giá trị biên -30, 30 -> [-30, 30, -900]
PASS ví dụ 2 của đề -> [0, 0, 9, 0, 0]
PASS 1..5 -> [120, 60, 40, 30, 24]
PASS một số lớn, còn lại là 1 -> [1, 5, 5, 5, 5, 5]


### 12.2. Tính chất và phản ví dụ

In [19]:
random.seed(238)

# Tính chất 1: nếu không có 0 thì nums[i] * answer[i] == tích cả mảng với mọi i
for _ in range(500):
    n = random.randint(2, 12)
    nums = [random.choice([x for x in range(-5, 6) if x != 0]) for _ in range(n)]
    ans = sol.productExceptSelf(nums)
    total = prod(nums)
    assert all(nums[i] * ans[i] == total for i in range(n))
print("PASS tính chất: nums[i] * answer[i] = tích cả mảng (khi không có 0)")

# Tính chất 2: số lượng số 0 quyết định số vị trí khác 0
for _ in range(500):
    n = random.randint(2, 12)
    nums = [random.randint(-3, 3) for _ in range(n)]
    ans = sol.productExceptSelf(nums)
    zeros = nums.count(0)
    if zeros >= 2:
        assert all(a == 0 for a in ans)
    elif zeros == 1:
        assert all(a == 0 for i, a in enumerate(ans) if nums[i] != 0)
print("PASS tính chất: từ hai số 0 mọi phần tử bằng 0; một số 0 thì chỉ vị trí số 0 có thể khác 0")

# Phản ví dụ: chia cho 0
def sai_chia(nums):
    total = prod(nums)
    return [total // x for x in nums]

try:
    sai_chia([-1, 1, 0, -3, 3])
except ZeroDivisionError as e:
    print("chia cho 0 ->", type(e).__name__)

PASS tính chất: nums[i] * answer[i] = tích cả mảng (khi không có 0)
PASS tính chất: từ hai số 0 mọi phần tử bằng 0; một số 0 thì chỉ vị trí số 0 có thể khác 0
chia cho 0 -> ZeroDivisionError


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [20]:
random.seed(2026)
for trial in range(3000):
    n = random.randint(2, 12)
    lo, hi = random.choice([(-3, 3), (-30, 30), (0, 2), (-1, 1)])
    nums = [random.randint(lo, hi) for _ in range(n)]
    expected = oracle(nums)
    for name, got in run_all(nums).items():
        assert got == expected, (name, nums, got, expected)

# mảng dài 10^5 (chỉ các cách O(n)); giá trị -1, 0, 1 để tích không phình to
for trial in range(5):
    n = 10**5
    zero_positions = set(random.sample(range(n), random.choice([0, 1, 2])))
    nums = [0 if i in zero_positions else random.choice([-1, 1]) for i in range(n)]
    results = {name: fn(list(nums)) for name, fn in impls.items() if name != "bruteforce"}
    base = results["two_pass"]
    for name, got in results.items():
        assert got == base, (name, trial)
print("PASS 3000 input ngẫu nhiên khớp oracle độc lập; 5 mảng 10^5 phần tử khớp giữa các cách O(n)")

PASS 3000 input ngẫu nhiên khớp oracle độc lập; 5 mảng 10^5 phần tử khớp giữa các cách O(n)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chia cho tích cả mảng (và quên số 0)

In [21]:
def sai_chia_tich(nums):
    total = prod(nums)
    return [total // x for x in nums]


print("[1,2,3,4] ->", sai_chia_tich([1, 2, 3, 4]), "(đúng nhưng đề CẤM chia)")
try:
    sai_chia_tich([-1, 1, 0, -3, 3])
except ZeroDivisionError as e:
    print("[-1,1,0,-3,3] -> ZeroDivisionError:", e)

[1,2,3,4] -> [24, 12, 8, 6] (đúng nhưng đề CẤM chia)
[-1,1,0,-3,3] -> ZeroDivisionError: integer division or modulo by zero


**Vì sao sai:** đề cấm phép chia; hơn nữa khi có số `0`, `total = 0` và chia cho `0` gây lỗi, còn khi chỉ cho `0` thì các vị trí khác bị tính sai. Cách dùng tiền tố/hậu tố chỉ nhân nên không gặp vấn đề này.

### Lỗi 2: Cập nhật `left` TRƯỚC khi gán (tích gồm cả `nums[i]`)

In [22]:
def sai_thu_tu(nums):
    n = len(nums)
    ans = [0] * n
    left = 1
    for i in range(n):
        left *= nums[i]                 # sai: cập nhật trước
        ans[i] = left                   # nay left đã gồm nums[i]
    right = 1
    for i in range(n - 1, -1, -1):
        ans[i] *= right
        right *= nums[i]
    return ans


print("sai:", sai_thu_tu([1, 2, 3, 4]), "| đúng:", Solution().productExceptSelf([1, 2, 3, 4]))

sai: [24, 24, 24, 24] | đúng: [24, 12, 8, 6]


**Vì sao sai:** `ans[i]` phải là tích **bên trái** `i`, tức **không** gồm `nums[i]`. Nếu nhân `nums[i]` vào `left` trước khi gán, `ans[i]` lấy luôn `nums[i]` và kết quả cuối bị nhân thừa `nums[i]`. Quy tắc: **gán rồi mới cập nhật**.

### Lỗi 3: Khởi tạo tích bằng `0` thay vì `1`

In [23]:
def sai_khoi_tao_0(nums):
    n = len(nums)
    ans = [0] * n
    left = 0                            # sai: 0 triệt tiêu mọi phép nhân
    for i in range(n):
        ans[i] = left
        left *= nums[i]
    right = 0
    for i in range(n - 1, -1, -1):
        ans[i] *= right
        right *= nums[i]
    return ans


print("sai:", sai_khoi_tao_0([1, 2, 3, 4]), "| đúng:", Solution().productExceptSelf([1, 2, 3, 4]))

sai: [0, 0, 0, 0] | đúng: [24, 12, 8, 6]


**Vì sao sai:** `1` là phần tử trung hòa của phép nhân (tích rỗng bằng `1`). Khởi tạo `0` làm mọi tích về `0`. Với phép **cộng** (tổng tiền tố) mới khởi tạo `0`.

### Lỗi 4: Lượt hậu tố chạy sai chiều

In [24]:
def sai_sai_chieu(nums):
    n = len(nums)
    ans = [0] * n
    left = 1
    for i in range(n):
        ans[i] = left
        left *= nums[i]
    right = 1
    for i in range(n):                   # sai: phải chạy từ phải sang trái
        ans[i] *= right
        right *= nums[i]
    return ans


print("sai:", sai_sai_chieu([1, 2, 3, 4]), "| đúng:", Solution().productExceptSelf([1, 2, 3, 4]))

sai: [1, 1, 4, 36] | đúng: [24, 12, 8, 6]


**Vì sao sai:** biến `right` phải chứa tích các phần tử **bên phải** `i`, nên phải duyệt từ **cuối về đầu**. Duyệt xuôi biến `right` lại tích các phần tử bên trái, cho kết quả như `left * left`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Kết quả tại `i` phụ thuộc mọi phần tử **trừ** `i`" và đề cấm chia → **tiền tố × hậu tố**.
- Mô-típ "hai lượt": lượt xuôi ghi thông tin **bên trái**, lượt ngược nhân thông tin **bên phải** bằng một biến chạy: bộ nhớ phụ `O(1)`.
- Luôn **gán trước, cập nhật sau** khi biến chạy phải *không gồm* phần tử hiện tại.
- Phần tử trung hòa: nhân thì `1`, cộng thì `0`; tích rỗng là `1`.
- Bài tương tự: tổng tiền tố/hậu tố (Prefix Sum), "Trapping Rain Water" (max trái/phải), "Candy", "Sum of Subarray Ranges".
- Với phép chia: số `0` luôn là trường hợp đặc biệt cần xử lý riêng (0, 1 hoặc ≥ 2 số `0`).

## 15. Biến thể của bài

1. **Tổng của mọi phần tử trừ `nums[i]`** (dễ hơn: `total − nums[i]`).
2. **Tích có modulo** `10⁹ + 7` (không chia được vì nghịch đảo modulo có thể không tồn tại): tiền tố/hậu tố vẫn dùng được.
3. **Ma trận:** `answer[i][j]` bằng tích mọi phần tử trừ `matrix[i][j]`.
4. **XOR/GCD/AND của mọi phần tử trừ `i`:** cùng khung tiền tố/hậu tố (không có phép chia để dùng).
5. **Tích của mọi phần tử trừ `nums[i]` và `nums[j]`** cho cặp `(i, j)`.

In [25]:
def sum_except_self(nums):
    """Biến thể 1."""
    total = sum(nums)
    return [total - x for x in nums]


def product_except_self_mod(nums, mod=10**9 + 7):
    """Biến thể 2: tích modulo, vẫn chỉ dùng tiền tố/hậu tố."""
    n = len(nums)
    ans = [1] * n
    left = 1
    for i in range(n):
        ans[i] = left
        left = left * nums[i] % mod
    right = 1
    for i in range(n - 1, -1, -1):
        ans[i] = ans[i] * right % mod
        right = right * nums[i] % mod
    return ans


from math import gcd


def gcd_except_self(nums):
    """Biến thể 4: ước chung lớn nhất của mọi phần tử trừ nums[i] (n >= 2), dùng tiền tố/hậu tố GCD."""
    n = len(nums)
    left = [0] * (n + 1)                   # gcd(0, x) = x nên 0 là phần tử trung hòa
    for i in range(n):
        left[i + 1] = gcd(left[i], nums[i])
    right = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        right[i] = gcd(right[i + 1], nums[i])
    return [gcd(left[i], right[i + 1]) for i in range(n)]


assert sum_except_self([1, 2, 3, 4]) == [9, 8, 7, 6]
assert product_except_self_mod([1, 2, 3, 4]) == [24, 12, 8, 6]
assert gcd_except_self([12, 18, 24]) == [6, 12, 6]
random.seed(15)
for _ in range(200):
    nums = [random.randint(-30, 30) for _ in range(random.randint(2, 12))]
    assert product_except_self_mod(nums) == [x % (10**9 + 7) for x in oracle(nums)]
    nums2 = [random.randint(1, 60) for _ in range(random.randint(2, 8))]
    assert gcd_except_self(nums2) == [gcd(*(nums2[:i] + nums2[i + 1:])) for i in range(len(nums2))]
print("PASS biến thể: tổng, modulo, gcd")

PASS biến thể: tổng, modulo, gcd


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Nhân trực tiếp | mỗi `i` nhân mọi `j ≠ i` | `O(n²)` | `O(1)` | hiểu đề; oracle (mảng nhỏ) |
| 2. Chia cho tích cả mảng | `total // nums[i]` + xử lý số 0 | `O(n)` | `O(1)` | **bị cấm** (đề cấm chia); chỉ để đối chiếu |
| 3. Hai mảng tiền tố/hậu tố | `left[i] * right[i]` | `O(n)` | `O(n)` | dễ hiểu nhất |
| 4. Hai lượt trên mảng kết quả | biến chạy `left`, `right` | `O(n)` | `O(1)` phụ | **cách nộp bài**, đáp ứng follow up |
| 5. Một vòng hai đầu | cập nhật `ans[i]` và `ans[n-1-i]` | `O(n)` | `O(1)` phụ | gọn, một vòng lặp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `prod([])`, `prod([2, 3, 4])`, `prod([5, 0, 7])`, `Solution().productExceptSelf([2, 3])`, `Solution().productExceptSelf([1, 0])`.

**Bài 2 (tính tay).** Với `nums = [2, 3, 4, 5]`: viết `left`, `right` và `answer`. Làm lại với `nums = [2, 0, 4, 5]` và `nums = [0, 0, 4]`.

**Bài 3 (điền TODO).** Hoàn thành `pes_todo(nums)` theo Cách 4 (hai lượt).

In [26]:
def pes_todo(nums):
    n = len(nums)
    ans = [0] * n
    left = 1
    # TODO 1: for i in range(n): gán ans[i] = left rồi left *= nums[i]
    right = 1
    # TODO 2: for i từ n-1 về 0: ans[i] *= right rồi right *= nums[i]
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `pes_arrays2(nums)` theo Cách 3 (hai mảng tiền tố/hậu tố) mà không nhìn code mẫu. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `sum_prefix_suffix(nums)` trả về mảng `answer[i]` = **tổng** mọi phần tử trừ `nums[i]` **mà không dùng `sum` của cả mảng** (dùng tiền tố/hậu tố). (b) Viết `product_except_two(nums, i, j)` trả về tích mọi phần tử trừ `nums[i]` và `nums[j]` bằng tiền tố/hậu tố (`i < j`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: tích rỗng là `1`; có số `0` thì tích về `0`. Bài 2: `answer[i] = tích bên trái × tích bên phải`.
- **Mức 2 (vừa):** Bài 1: `1`, `24`, `0`, `[3, 2]`, `[0, 1]`. Bài 2: `[2,3,4,5]`: `left = [1,2,6,24]`, `right = [60,20,5,1]`, `answer = [60,40,30,24]`. `[2,0,4,5]`: `[0,40,0,0]`. `[0,0,4]`: `[0,0,0]`. Bài 3: hai TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `left[i] = left[i-1]*nums[i-1]`, `right[i] = right[i+1]*nums[i+1]`. Bài 5(a): `left` là tổng tiền tố, `right` tổng hậu tố, `answer[i] = left[i] + right[i]`. Bài 5(b): `prefix(0..i-1) × giữa(i+1..j-1) × suffix(j+1..)`; dễ nhất là dùng `prod` của ba lát cắt (hoặc dựng tiền tố và hậu tố).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert prod([]) == 1 and prod([2, 3, 4]) == 24 and prod([5, 0, 7]) == 0
assert Solution().productExceptSelf([2, 3]) == [3, 2]
assert Solution().productExceptSelf([1, 0]) == [0, 1]

# Bài 2
assert Solution().productExceptSelf([2, 3, 4, 5]) == [60, 40, 30, 24]
assert Solution().productExceptSelf([2, 0, 4, 5]) == [0, 40, 0, 0]
assert Solution().productExceptSelf([0, 0, 4]) == [0, 0, 0]

# Bài 3
def pes_todo_ans(nums):
    n = len(nums)
    ans = [0] * n
    left = 1
    for i in range(n):
        ans[i] = left
        left *= nums[i]
    right = 1
    for i in range(n - 1, -1, -1):
        ans[i] *= right
        right *= nums[i]
    return ans

for nums, expected in examples:
    assert pes_todo_ans(nums) == expected

# Bài 4
def pes_arrays2(nums):
    n = len(nums)
    left = [1] * n
    for i in range(1, n):
        left[i] = left[i - 1] * nums[i - 1]
    right = [1] * n
    for i in range(n - 2, -1, -1):
        right[i] = right[i + 1] * nums[i + 1]
    return [a * b for a, b in zip(left, right)]

for nums, expected in examples:
    assert pes_arrays2(nums) == expected

# Bài 5(a), 5(b)
def sum_prefix_suffix(nums):
    n = len(nums)
    left = [0] * n
    for i in range(1, n):
        left[i] = left[i - 1] + nums[i - 1]
    right = [0] * n
    for i in range(n - 2, -1, -1):
        right[i] = right[i + 1] + nums[i + 1]
    return [a + b for a, b in zip(left, right)]

def product_except_two(nums, i, j):
    assert i < j
    return prod(nums[:i]) * prod(nums[i + 1:j]) * prod(nums[j + 1:])

assert sum_prefix_suffix([1, 2, 3, 4]) == [9, 8, 7, 6]
assert product_except_two([1, 2, 3, 4, 5], 1, 3) == 1 * 3 * 5
random.seed(19)
for _ in range(200):
    nums = [random.randint(-9, 9) for _ in range(random.randint(2, 10))]
    assert sum_prefix_suffix(nums) == sum_except_self(nums)
    if len(nums) >= 2:
        i, j = sorted(random.sample(range(len(nums)), 2))
        assert product_except_two(nums, i, j) == prod([x for k, x in enumerate(nums) if k not in (i, j)])
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai lượt: tích bên trái rồi nhân tích bên phải)

```python
class Solution:
    def productExceptSelf(self, nums: List[int]) -> List[int]:
        n = len(nums)
        ans = [0] * n
        left = right = 1
        for i in range(n):
            ans[i] = left
            left *= nums[i]
        for i in range(n - 1, -1, -1):
            ans[i] *= right
            right *= nums[i]
        return ans
```

**Checklist**

- [ ] **Không** dùng phép chia (và không phải xử lý riêng số `0`)?
- [ ] `left` và `right` khởi tạo bằng **`1`** (tích rỗng), không phải `0`?
- [ ] **Gán trước, cập nhật sau**: `ans[i] = left` rồi mới `left *= nums[i]`?
- [ ] Lượt thứ hai chạy **từ phải sang trái** (`range(n - 1, -1, -1)`)?
- [ ] Bộ nhớ phụ `O(1)` (mảng kết quả không tính), thời gian `O(n)`?
- [ ] Đã thử: `n = 2`, có một số `0`, có hai số `0`, số âm, toàn số `1`?
- [ ] Đã đối chiếu với `prod` của mảng bỏ phần tử `i` trên input ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def productExceptSelf(self, nums: List[int]) -> List[int]`?